# Week 1 — SQL diagnostic and loan-portfolio foundations

**Learn in Colab. Save evidence in GitHub.** This workbook starts section 4 of your curriculum: advanced SQL and database engineering.

**Business problem:** Describe the March 2025 loan portfolio accurately, measure a defined 30+ DPD balance share, and relate balances to cash receipts without double-counting.

**Today:** 20 minutes for setup, 75 minutes for a first-attempt diagnostic, and 25 minutes to record gaps. The full week is 20 hours.

All records are synthetic teaching data, with money in KES. DuckDB runs inside the notebook; no database account, Drive mount, or GPU is needed.

- [Week 1 plan](https://github.com/dianawambugha/Multi-Sector-Career-Optimisation/blob/main/weeks/week-01/README.md)
- [Progress tracker](https://github.com/dianawambugha/Multi-Sector-Career-Optimisation/blob/main/progress/weekly-log.md)
- [Data dictionary](https://github.com/dianawambugha/Multi-Sector-Career-Optimisation/blob/main/weeks/week-01/data/README.md)

Your first attempt is a baseline, not a pass/fail judgment about your career readiness. Attempt the questions before reading the later lesson notes or reference SQL.

## 1. Connect and install

Select a Python 3 **CPU** runtime and run each setup cell in order with its play button. The first cell installs a fixed DuckDB version. Colab already provides pandas.

At the end of each session use **File → Save a copy in GitHub**. Select `dianawambugha/Multi-Sector-Career-Optimisation`, branch `main`, and path `weeks/week-01/week-01-sql.ipynb`. Update the Markdown progress tracker separately on GitHub.

In [ ]:
%pip install -q duckdb==1.4.3

In [ ]:
from io import StringIO
import duckdb
import pandas as pd
from IPython.display import display

# Embedded copies of the repository CSV fixtures keep Colab self-contained.
CSV_DATA = {'customers': 'customer_id,county,segment\nC001,Nairobi,Salaried\nC002,Mombasa,Self-employed\nC003,Kisumu,Microbusiness\nC004,Nairobi,SME\nC005,Nakuru,Salaried\nC006,Uasin Gishu,Microbusiness\nC007,Nairobi,Self-employed\n', 'loans': 'loan_id,customer_id,product,disbursement_date,original_principal\nL001,C001,Personal,2024-10-10,200000\nL002,C002,Personal,2024-11-01,180000\nL003,C003,Business,2024-12-01,500000\nL004,C004,Business,2024-12-15,600000\nL005,C001,Asset,2024-12-20,300000\nL006,C005,Personal,2025-01-05,100000\nL007,C006,Business,2025-02-10,200000\nL008,C007,Personal,2025-03-05,100000\nL009,C002,Asset,2025-01-10,200000\n', 'payments': 'payment_id,loan_id,payment_date,payment_amount\nP001,L001,2025-01-20,15000\nP002,L001,2025-02-20,15000\nP003,L001,2025-03-05,7500\nP004,L001,2025-03-20,7500\nP005,L002,2025-03-10,10000\nP006,L003,2025-03-15,30000\nP007,L004,2025-03-15,20000\nP008,L006,2025-03-28,80000\nP009,L007,2025-03-29,10000\nP010,L008,2025-04-02,9000\nP011,L009,2025-03-05,10000\n', 'snapshots': 'loan_id,snapshot_date,outstanding_balance,dpd\nL001,2025-01-31,180000,0\nL002,2025-01-31,170000,15\nL003,2025-01-31,480000,0\nL004,2025-01-31,580000,30\nL005,2025-01-31,290000,0\nL006,2025-01-31,100000,0\nL009,2025-01-31,180000,0\nL001,2025-02-28,165000,0\nL002,2025-02-28,160000,30\nL003,2025-02-28,450000,0\nL004,2025-02-28,560000,60\nL005,2025-02-28,280000,5\nL007,2025-02-28,200000,0\nL001,2025-03-31,150000,0\nL002,2025-03-31,150000,45\nL003,2025-03-31,420000,0\nL004,2025-03-31,540000,90\nL005,2025-03-31,270000,30\nL006,2025-03-31,0,0\nL007,2025-03-31,190000,15\nL008,2025-03-31,90000,0\n', 'staging_snapshots': 'loan_id,snapshot_date,outstanding_balance,dpd\nL001,2025-03-31,150000,0\nL001,2025-03-31,150000,0\nL002,2025-03-31,-100,30\nL999,2025-03-31,1000,0\nL003,2025-03-31,420000,\nL004,2025-03-31,540000,-1\n'}

def build_database():
    database = duckdb.connect(':memory:')
    for name, csv_text in CSV_DATA.items():
        frame = pd.read_csv(StringIO(csv_text))
        for column in frame.columns:
            if column.endswith('_date'):
                frame[column] = pd.to_datetime(frame[column])
        database.register('_incoming', frame)
        expressions = []
        for column in frame.columns:
            if column.endswith('_date'):
                expressions.append(f'CAST({column} AS DATE) AS {column}')
            elif column in {'original_principal', 'outstanding_balance', 'payment_amount'}:
                expressions.append(f'CAST({column} AS DECIMAL(18,2)) AS {column}')
            elif column == 'dpd':
                expressions.append('CAST(dpd AS INTEGER) AS dpd')
            else:
                expressions.append(column)
        database.execute(f'CREATE TABLE {name} AS SELECT {", ".join(expressions)} FROM _incoming')
        database.unregister('_incoming')
    return database

if 'con' in globals():
    con.close()
con = build_database()

def sql(query):
    return con.execute(query).df()

print(f'DuckDB {duckdb.__version__}; five synthetic tables loaded into memory.')
display(sql('SHOW TABLES'))

## 2. Understand the grain before querying

| Table | Rows | One row represents | Key |
| --- | ---: | --- | --- |
| `customers` | 7 | One customer | `customer_id` |
| `loans` | 9 | One loan account | `loan_id` |
| `snapshots` | 21 | One loan on one reporting date | `(loan_id, snapshot_date)` |
| `payments` | 11 | One cash payment | `payment_id` |
| `staging_snapshots` | 6 | One unvalidated incoming record | Intended snapshot key; intentionally violated |

A customer can have several loans. A loan can have several snapshots and payments. `outstanding_balance` is remaining principal; `payment_amount` is gross cash and is not split into principal, interest, or fees. `dpd` means days past due.

March reporting means **snapshot_date = DATE '2025-03-31'**, including the zero-balance account. Missing snapshots are not zero balances. The staging table is only for the data-quality exercise; keep it out of portfolio totals.

Run these integrity checks against the trusted main tables. These checks validate the fixture, not your diagnostic answers.

In [ ]:
expected_counts = {'customers': 7, 'loans': 9, 'snapshots': 21, 'payments': 11, 'staging_snapshots': 6}
for table, expected_count in expected_counts.items():
    assert con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0] == expected_count, table

quality_checks = {
    'unique customer IDs': 'SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM customers',
    'unique loan IDs': 'SELECT COUNT(*) - COUNT(DISTINCT loan_id) FROM loans',
    'unique payment IDs': 'SELECT COUNT(*) - COUNT(DISTINCT payment_id) FROM payments',
    'unique loan-date snapshots': """
        SELECT COUNT(*) FROM (
            SELECT loan_id, snapshot_date FROM snapshots
            GROUP BY loan_id, snapshot_date HAVING COUNT(*) > 1
        ) repeated_keys
    """,
    'valid loan customers': """
        SELECT COUNT(*) FROM loans l LEFT JOIN customers c USING (customer_id)
        WHERE c.customer_id IS NULL
    """,
    'valid snapshot loans': """
        SELECT COUNT(*) FROM snapshots s LEFT JOIN loans l USING (loan_id)
        WHERE l.loan_id IS NULL
    """,
    'valid payment loans': """
        SELECT COUNT(*) FROM payments p LEFT JOIN loans l USING (loan_id)
        WHERE l.loan_id IS NULL
    """,
    'valid main balances and DPD': """
        SELECT COUNT(*) FROM snapshots
        WHERE outstanding_balance < 0 OR outstanding_balance IS NULL OR dpd < 0 OR dpd IS NULL
    """
}
for name, query in quality_checks.items():
    assert con.execute(query).fetchone()[0] == 0, name
print('PASS: fixture row counts and eight main-table integrity checks.')
print('The staging table intentionally contains defects for Q8.')

### Demonstration: inspect observations, without aggregating

Run this small query and inspect its column names. For your answers, edit the text between triple quotes in the question cells. Python only sends that SQL to DuckDB.

In [ ]:
display(sql("""
SELECT loan_id, snapshot_date, outstanding_balance, dpd
FROM snapshots
WHERE snapshot_date = DATE '2025-01-31'
ORDER BY loan_id;
"""))

## 3. Start your diagnostic — 75 minutes

Run the answer-check setup cell once, then attempt Q1–Q8. Use the exact output column names in each prompt. Row order is ignored by the checker, but use an explicit `ORDER BY` for readable results. Numeric answers are checked to half a cent (or half of 0.01 percentage point).

Each question has an **answer cell** and a **check cell**. A blank answer is marked **Not attempted**. Your first nonblank checked submission is retained during this runtime; later submissions update the retry result. Copy the first-attempt results into GitHub before restarting, because runtime memory is temporary.

The folded cell contains expected results. Leave it folded during your diagnostic. Result checks use one small fixture; they do not assess SQL reasoning or prove correctness for every possible dataset.

In [ ]:
# @title Set up answer checks (run; leave folded during the diagnostic)
from pandas.testing import assert_frame_equal

EXPECTED = {'Q1': {'columns': ['loan_count', 'customer_count', 'total_outstanding'], 'rows': [[8, 7, 1810000]], 'keys': ['loan_count']}, 'Q2': {'columns': ['product', 'loan_count', 'total_outstanding', 'average_outstanding'], 'rows': [['Asset', 1, 270000, 270000], ['Business', 3, 1150000, 383333.33], ['Personal', 4, 390000, 97500]], 'keys': ['product']}, 'Q3': {'columns': ['loan_id', 'snapshot_date', 'outstanding_balance'], 'rows': [['L001', '2025-03-31', 150000], ['L002', '2025-03-31', 150000], ['L003', '2025-03-31', 420000], ['L004', '2025-03-31', 540000], ['L005', '2025-03-31', 270000], ['L006', '2025-03-31', 0], ['L007', '2025-03-31', 190000], ['L008', '2025-03-31', 90000], ['L009', '2025-01-31', 180000]], 'keys': ['loan_id']}, 'Q4': {'columns': ['product', 'total_outstanding', 'overdue_30plus_balance', 'overdue_30plus_pct'], 'rows': [['Asset', 270000, 270000, 100], ['Business', 1150000, 540000, 46.96], ['Personal', 390000, 150000, 38.46]], 'keys': ['product']}, 'Q5': {'columns': ['loan_id', 'snapshot_date', 'previous_snapshot_date', 'outstanding_balance', 'previous_balance', 'balance_change'], 'rows': [['L001', '2025-03-31', '2025-02-28', 150000, 165000, -15000], ['L002', '2025-03-31', '2025-02-28', 150000, 160000, -10000], ['L003', '2025-03-31', '2025-02-28', 420000, 450000, -30000], ['L004', '2025-03-31', '2025-02-28', 540000, 560000, -20000], ['L005', '2025-03-31', '2025-02-28', 270000, 280000, -10000], ['L006', '2025-03-31', '2025-01-31', 0, 100000, -100000], ['L007', '2025-03-31', '2025-02-28', 190000, 200000, -10000], ['L008', '2025-03-31', None, 90000, None, None]], 'keys': ['loan_id']}, 'Q6': {'columns': ['product', 'loan_id', 'outstanding_balance', 'position'], 'rows': [['Asset', 'L005', 270000, 1], ['Business', 'L004', 540000, 1], ['Business', 'L003', 420000, 2], ['Personal', 'L001', 150000, 1], ['Personal', 'L002', 150000, 2]], 'keys': ['product', 'position']}, 'Q7': {'columns': ['loan_id', 'outstanding_balance', 'march_payments'], 'rows': [['L001', 150000, 15000], ['L002', 150000, 10000], ['L003', 420000, 30000], ['L004', 540000, 20000], ['L005', 270000, 0], ['L006', 0, 80000], ['L007', 190000, 10000], ['L008', 90000, 0]], 'keys': ['loan_id']}, 'Q8': {'columns': ['issue', 'issue_count'], 'rows': [['duplicate_key_groups', 1], ['orphan_loan_rows', 1], ['negative_balance_rows', 1], ['missing_dpd_rows', 1], ['negative_dpd_rows', 1]], 'keys': ['issue']}}
POINTS = {'Q1': 10, 'Q2': 10, 'Q3': 15, 'Q4': 15, 'Q5': 15, 'Q6': 10, 'Q7': 15, 'Q8': 10}
TEXT_COLUMNS = {'product', 'loan_id', 'issue'}
answers = globals().get('answers', {})
first_results = globals().get('first_results', {})
first_answers = globals().get('first_answers', {})
latest_results = globals().get('latest_results', {})

def normalized(frame, spec):
    if len(frame.columns) != len(spec['columns']) or set(frame.columns) != set(spec['columns']):
        raise AssertionError(f"Use these exact output column names: {spec['columns']}; received {list(frame.columns)}")
    result = frame.loc[:, spec['columns']].copy()
    for column in result.columns:
        if column.endswith('_date'):
            result[column] = pd.to_datetime(result[column], errors='raise').dt.strftime('%Y-%m-%d').astype('string')
        elif column in TEXT_COLUMNS:
            result[column] = result[column].astype('string')
        else:
            result[column] = pd.to_numeric(result[column], errors='raise').astype('float64')
    return result.sort_values(spec['keys'], na_position='last').reset_index(drop=True)

def check_answer(question, query=None, show=True):
    query = answers.get(question, '') if query is None else query
    if not query.strip():
        latest_results[question] = {'status': 'Not attempted', 'points': 0}
        if show:
            print(f'{question}: Not attempted. Write your SQL, run its answer cell, then run this check.')
        return None
    try:
        statements = con.extract_statements(query)
        if len(statements) != 1 or statements[0].type != duckdb.StatementType.SELECT:
            raise ValueError('Submit one SELECT statement; WITH ... SELECT is supported.')
        actual = con.execute(query).df()
        spec = EXPECTED[question]
        expected_frame = pd.DataFrame(spec['rows'], columns=spec['columns'])
        assert_frame_equal(normalized(actual, spec), normalized(expected_frame, spec),
                           check_dtype=False, check_exact=False, rtol=0, atol=0.005)
    except Exception as error:
        result = {'status': 'Needs review', 'points': 0}
        if show:
            print(f'{question}: needs review. {type(error).__name__}: {error}')
    else:
        result = {'status': 'Passed', 'points': POINTS[question]}
        if show:
            print(f'{question}: result check passed ({POINTS[question]} points). Explain why your query works.')
            display(actual)
    first_results.setdefault(question, result.copy())
    first_answers.setdefault(question, query)
    latest_results[question] = result
    return result['status'] == 'Passed'

def diagnostic_summary():
    rows = []
    for question, possible in POINTS.items():
        first = first_results.get(question, {'status': 'Not attempted', 'points': 0})
        current = latest_results.get(question, {'status': 'Not attempted', 'points': 0})
        rows.append({'question': question, 'maximum': possible,
                     'first_status': first['status'], 'first_points': first['points'],
                     'latest_status': current['status'], 'latest_points': current['points']})
    summary = pd.DataFrame(rows)
    display(summary)
    print(f'First checked submissions: {len(first_results)}/8; earned {summary.first_points.sum()}/100 points.')
    print(f'Latest checks: earned {summary.latest_points.sum()}/100 points. Record whether you used help.')
    if not first_results:
        print('No diagnostic questions attempted yet; this is not a completed assessment.')
    return summary

print('Answer checks ready. Empty answers will remain Not attempted.')

## Q1. Portfolio grain — 10 points

For **2025-03-31**, return one row with:

- `loan_count`: number of loan accounts with a March snapshot, including zero balance.
- `customer_count`: distinct customers represented by those accounts.
- `total_outstanding`: sum of their outstanding balances, in KES.

Use `snapshots` and `loans`. Explain afterward why a customer count can differ from a loan count.

In [ ]:
answers["Q1"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q1");

## Q2. Portfolio by product — 10 points

For March snapshots, return `product`, `loan_count`, `total_outstanding`, and `average_outstanding` for each product. Include the zero-balance account. Round the average to two decimal places.

Explain afterward why averaging product averages without weighting can misstate the portfolio average.

In [ ]:
answers["Q2"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q2");

## Q3. Latest available snapshot — 15 points

Use `ROW_NUMBER` in a CTE to return the latest available observation for **every loan with snapshots**. Output `loan_id`, `snapshot_date`, and `outstanding_balance`.

Do not restrict the input to March. Explain why these rows are not necessarily a same-date portfolio report. `(loan_id, snapshot_date)` is unique in the trusted table.

In [ ]:
answers["Q3"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q3");

## Q4. 30+ DPD balance share — 15 points

Use a CTE to calculate these March measures by product:

- `product`
- `total_outstanding`
- `overdue_30plus_balance`: outstanding principal where **dpd >= 30**
- `overdue_30plus_pct`: `100 × overdue_30plus_balance / total_outstanding`, rounded to two decimals

Use `NULLIF` for a zero denominator. The output is a percentage on a 0–100 scale. This is a **balance-weighted** measure; use the definition above, including exactly 30 DPD. Explain why this is different from the percentage of accounts with 30+ DPD.

In [ ]:
answers["Q4"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q4");

## Q5. Change from the previous observation — 15 points

For each March snapshot, show `loan_id`, `snapshot_date`, `previous_snapshot_date`, `outstanding_balance`, `previous_balance`, and `balance_change` (current minus previous).

Use `LAG` partitioned by loan and ordered by snapshot date. Compute the windows across **all observations**, then filter the output to March. Keep missing previous values as null. Explain the difference between the previous observed row and the previous calendar month.

In [ ]:
answers["Q5"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q5");

## Q6. Top two loans per product — 10 points

Among March loans with **outstanding_balance > 0**, return up to two loans per product. Output `product`, `loan_id`, `outstanding_balance`, and `position`.

Use `ROW_NUMBER` ordered by balance descending, then loan ID ascending within each product. Return positions 1 and 2 where available. This task asks for exactly two when possible; it does not ask to include every tie. Explain how `RANK` or `DENSE_RANK` could change the result.

In [ ]:
answers["Q6"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q6");

## Q7. Join collections without duplicating balances — 15 points

For every loan with a March snapshot, return `loan_id`, `outstanding_balance`, and `march_payments`.

Payments qualify when `payment_date >= DATE '2025-03-01'` and `payment_date < DATE '2025-04-01'`. Aggregate payments by loan **before** joining to snapshots. Preserve accounts with no qualifying payments and show zero for their receipts.

Do not use `SUM(DISTINCT payment_amount)`; equal-value payments can be separate real transactions. Explain why an account with no March snapshot is outside this report even if it has a March payment.

In [ ]:
answers["Q7"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q7");

## Q8. Quarantined data-quality issues — 10 points

Inspect `staging_snapshots` only; do not repair or mix it into the trusted portfolio. Return `issue` and `issue_count`, with one row for **each** of the following labels:

- `duplicate_key_groups`: the number of `(loan_id, snapshot_date)` groups with more than one row.
- `orphan_loan_rows`: records whose loan ID does not exist in `loans`.
- `negative_balance_rows`: records with a negative balance.
- `missing_dpd_rows`: records with null DPD.
- `negative_dpd_rows`: records with negative DPD.

Combine checks using `UNION ALL`. Explain why adding these counts does not necessarily produce a count of unique bad records.

In [ ]:
answers["Q8"] = """

"""
# Write your SQL between the triple quotes above.

In [ ]:
check_answer("Q8");

## 4. Record your baseline before studying the answers

Run the summary. Copy your first-attempt scores and confidence into `progress/weekly-log.md`; keep them unchanged when you retry. Save the notebook to GitHub so your SQL and results form part of your learning record.

The first-answer snapshot below captures the SQL you originally checked during this runtime. Saving outputs helps preserve that evidence, but copy the score to the tracker before resetting the runtime.

In [ ]:
summary = diagnostic_summary()
for question, original_sql in first_answers.items():
    print(f'\n{question} — first checked SQL:\n{original_sql.strip()}')

### Your reflection — edit this Markdown cell

- Time spent:
- Questions I attempted independently:
- Questions I left unanswered:
- My hardest concept:
- One mistake and why it happened:
- What evidence makes me trust my answer:
- A business limitation I should communicate:
- My next practice task:

For each completed question, add a one- or two-sentence explanation of its grain and method. Result checks cannot write that explanation for you.

## 5. Lessons for Days 2–4 — read after the diagnostic

### Grain and join multiplication

Grain means what **one row represents**. A monthly balance is observed once per loan per reporting date. A payment table has one row per transaction. If a loan has two qualifying payments, a direct join repeats its balance twice. Pre-aggregate the payments to one row per loan, then join the two tables at compatible grain.

### CTEs make the stages explicit

A CTE names an intermediate result. It can separate filtering, grouping, and final calculations. A CTE is not a promise that the database will materialize that result or run faster. Read the example below as two stages: calculate January product balances, then select the larger groups. This is a demonstration, not one of your scored questions.

In [ ]:
display(sql("""
WITH january_products AS (
    SELECT l.product, SUM(s.outstanding_balance) AS balance
    FROM snapshots s
    JOIN loans l ON l.loan_id = s.loan_id
    WHERE s.snapshot_date = DATE '2025-01-31'
    GROUP BY l.product
)
SELECT product, balance
FROM january_products
WHERE balance >= 400000
ORDER BY balance DESC;
"""))

### Windows preserve rows

`GROUP BY` reduces records to group-level rows. A window function can add a calculation while retaining the underlying rows. `PARTITION BY loan_id` restarts a calculation for each loan. The window's `ORDER BY snapshot_date` defines its sequence; an outer `ORDER BY` controls display order.

`ROW_NUMBER` assigns a unique position; give tied values a stable secondary key. `LAG` reads the previous observed row. If you filter to March **before** calculating `LAG`, earlier observations are gone. A useful structure is: compute windows in a CTE, then filter in the outer query.

### Define the metric before writing SQL

Specify the population, reporting date, numerator, denominator, units, and treatment of missing data. A balance-weighted delinquency share answers a different question from a count-based share. Neither is a default probability or an expected credit loss estimate.

### Oracle and PostgreSQL transfer notes

| Concept | This workbook | Transfer consideration |
| --- | --- | --- |
| CTEs | `WITH ... AS (...)` | Familiar structure in Oracle and PostgreSQL; optimization behavior differs |
| Windows | `ROW_NUMBER`, `LAG`, `OVER` | Same core ideas; declare ordering and null behavior deliberately |
| Date literals | `DATE '2025-03-31'` | Supported in all three; date arithmetic and truncation functions differ |
| Null handling | `COALESCE`, `NULLIF` | Portable for these exercises; Oracle also has `NVL` |
| Conditional sums | `SUM(CASE WHEN ... THEN ... ELSE 0 END)` | Portable approach used here |
| Plans and indexes | DuckDB `EXPLAIN` | Operators and index behavior differ from Oracle/PostgreSQL; do not transfer performance claims blindly |

After trying your own repair, compare with the [reference SQL](https://github.com/dianawambugha/Multi-Sector-Career-Optimisation/blob/main/weeks/week-01/solutions.sql). Re-run your answer cell and check cell. Then close the reference and explain the query unaided.

## 6. Day 6 — inspect two execution plans

Once Q2 and Q7 pass, inspect their plans. Identify the scan, grouping, and join operators. For Q7, describe where the payment aggregation appears. Plans are estimates/strategies; this tiny dataset cannot support a production-speed claim. Engine-specific index and partitioning work comes later.

In [ ]:
for question in ('Q2', 'Q7'):
    if latest_results.get(question, {}).get('status') == 'Passed':
        print(f'\n{question}: EXPLAIN')
        for key, value in con.execute('EXPLAIN ' + answers[question]).fetchall():
            print(value)
    else:
        print(f'{question}: finish the exercise before inspecting its plan.')

### Your plan notes — edit this cell

- Q2: scan and aggregation operators, and what rows they consume:
- Q7: join type and location of payment aggregation:
- One question that requires a larger dataset or a different database engine:

## 7. Finish the week with evidence

1. Write a 150–200 word decision brief with date, units, checked findings, recommended next step, and a limitation.
2. Record a three-minute explanation of the business problem, method, finding, and implication.
3. Complete the [resume and communication worksheet](https://github.com/dianawambugha/Multi-Sector-Career-Optimisation/blob/main/weeks/week-01/career-and-communication.md).
4. Save your baseline, then start a fresh runtime and run the completed notebook in order. Empty answer cells do not count as passed tests.
5. Save your notebook in GitHub and update the [progress log](https://github.com/dianawambugha/Multi-Sector-Career-Optimisation/blob/main/progress/weekly-log.md) with actual hours and evidence links.
6. Independently retry two questions and record whether you needed help. Use the gaps to plan Week 2.

**Your first next step today:** finish Q1, save your attempt, and explain why a loan count differs from a customer count.